# Single-cell EMI

## Modelling of excitable cells - the EMI Model

The **Extracellular-Membrane-Intracellular (EMI)** model is a detailed mathematical framework that explicitly represents the intracellular space ($\Omega_i$), extracellular space ($\Omega_e$), and the cell membrane ($\Gamma$) as separate physical domains. Unlike homogenized models such as the bidomain and monodomain models, the EMI model resolves the actual cellular geometry and treats the membrane as a true interface where electrical coupling occurs. This makes it particularly useful for studying microscale phenomena that cannot be captured by simpler continuum descriptions. A comprehensive introduction to the EMI framework is given in the book *Modeling Excitable Tissue: The EMI Framework* by Tveito et al. (2021) {cite}`emisinglecell-tveito2021modeling`.


### The single-cell model
In this notebook, we restrict attention to a single cell surrounded by an extracellular space. In this setting, the EMI model is given by the following equations

$$
\begin{align*}
     -\nabla \cdot (\sigma_e\nabla u_e) &= f_e&& \text{in } \Omega_e \tag{1},\\
     -\nabla \cdot (\sigma_i\nabla u_i) &= f_i&& \text{in } \Omega_i \tag{2},\\
     \sigma_e\nabla u_e\cdot \mathbf{n}_e = - \sigma_i\nabla u_i\cdot \mathbf{n}_i &\equiv I_m&&\text{at } \Gamma \tag{3},\\
     v &=u_i-u_e&& \text{at } \Gamma \tag{4},\\
     \frac{\partial v}{\partial t} &= \frac{1}{C_m}(I_m-I_{ion})&& \text{at } \Gamma \tag{5},
\end{align*}
$$
where:
* $u_i, u_e$: Intracellular and extracellular potentials.
* $v$: Transmembrane potential.
* $I_m$: Transmembrane current (enforcing current conservation across the membrane interface).
* $\sigma_i, \sigma_e$: Intracellular and extracellular conductivities.
* $C_m$: Membrane capacitance.
* $I_{\text{ion}}(v)$: Ionic current, which depends on the specific ionic model chosen.
* $f_i, f_e$: Source terms. These are typically zero, though we will later use $f_i$ to inject a stimulus current.


#### Boundary and initial conditions
We close the EMI problem by applying homogeneous boundary conditions on the outer boundary:

$$
\begin{align*}
     u_e(\mathbf{x},t) = 0 & \quad \mathrm{for} \quad \mathbf{x}\in\partial\Omega_e^D,\\ 
     \sigma_e \nabla u_e (\mathbf{x},t) \cdot \mathbf{n}_e = 0 & \quad \mathrm{for} \quad \mathbf{x}\in\partial\Omega_e^N,
\end{align*}
$$

along with an initial membrane potential $v(\mathbf{x},0)=v_0(\mathbf{x})$.

\
![emi_domain](figures/emi_domain_singlecell.png)

*Illustration of the intracellular and extracellular space. Figure from {cite}`emisinglecell-tveito2021modeling`.*

\
**Note:** The implementation in this notebook is based on [a tutorial](https://scientificcomputing.github.io/fenics-in-the-wild/src/ucs/emi/emi_primal_single.html) by Jørgen Dokken.


In [ ]:
from pathlib import Path

from mpi4py import MPI
from dolfinx import fem, mesh, io, plot, default_scalar_type
from ufl import (
    inner,
    grad,
    TestFunctions,
    TrialFunctions,
    MixedFunctionSpace,
    extract_blocks,
    Measure,
    SpatialCoordinate,
    conditional,
    And,
)
import numpy as np
import scifem
import pyvista
pyvista.set_jupyter_backend('static')

## Geometry – Separating the intra- and extracellular 
EMI simulations require a computational domain that explicitly resolves both the intracellular and extracellular regions. In this example, we construct a simple "square-in-a-square" geometry. 

We define the intracellular domain as $\Omega_i = [0.25, 0.75] \times [0.25, 0.75]$ sitting inside a larger unit square $[0,1]^2$. The extracellular domain is simply the remaining complement, $\Omega_e = [0,1]^2 \setminus \Omega_i$. 

To implement this, we first define a few helper functions to mathematically identify the interior boundaries.

In [ ]:
x_L = 0.25
x_U = 0.75
y_L = 0.25
y_U = 0.75


def lower_bound(x, i, bound, tol=1e-12):
    return x[i] >= bound - tol


def upper_bound(x, i, bound, tol=1e-12):
    return x[i] <= bound + tol


def omega_interior_marker(x, tol=1e-12):
    return (
        lower_bound(x, 0, x_L, tol=tol)
        & lower_bound(x, 1, y_L, tol=tol)
        & upper_bound(x, 0, x_U, tol=tol)
        & upper_bound(x, 1, y_U, tol=tol)
    )

We can now generate the full computational domain $\Omega$ as a standard 2D unit square mesh.

In [ ]:
M = 50
omega = mesh.create_unit_square(
    MPI.COMM_WORLD, M, M, ghost_mode=mesh.GhostMode.shared_facet
)

To distinguish between the two spaces, we create a {py:class}`MeshTags<dolfinx.mesh.MeshTags>` object. Cells belonging to the intracellular region are tagged with `interior_marker`, while all remaining cells are tagged with `exterior_marker`.

In [ ]:
interior_cells = mesh.locate_entities(omega, omega.topology.dim, omega_interior_marker)

interior_marker = 2
exterior_marker = 3

cell_map = omega.topology.index_map(omega.topology.dim)
num_cells_local = cell_map.size_local + cell_map.num_ghosts

cell_marker = np.full(num_cells_local, exterior_marker, dtype=np.int32)
cell_marker[interior_cells] = interior_marker

ct = mesh.meshtags(
    omega, omega.topology.dim, np.arange(num_cells_local, dtype=np.int32), cell_marker
)

Next, we construct separate {py:class}`Mesh<dolfinx.mesh.Mesh>` objects for the intracellular and extracellular domains by extracting submeshes from the parent mesh using the [`scifem.extract_submesh()`](https://scientificcomputing.github.io/scifem/docs/api.html#scifem.extract_submesh) utility.

In [ ]:
omega_i, interior_to_parent, _, _, _ = scifem.extract_submesh(
    omega, ct, interior_marker
)
omega_e, exterior_to_parent, _, _, _ = scifem.extract_submesh(
    omega, ct, exterior_marker
)

We identify the facets on the interface $\Gamma$ using [`scifem.find_interface()`](https://scientificcomputing.github.io/scifem/docs/api.html#scifem.find_interface).


In [ ]:
gamma_facets = scifem.find_interface(ct, interior_marker, exterior_marker)

Finally, we create a second {py:class}`MeshTags<dolfinx.mesh.MeshTags>` object to mark these
interior facets on $\Gamma$ with `interface_marker`, and all exterior boundary facets with `boundary_marker`.
This will allow us to accurately apply boundary conditions and calculate interface integrals later.

In [ ]:
omega.topology.create_connectivity(omega.topology.dim - 1, omega.topology.dim)
exterior_facets = mesh.exterior_facet_indices(omega.topology)
facet_map = omega.topology.index_map(omega.topology.dim - 1)
num_facets_local = facet_map.size_local + facet_map.num_ghosts
facets = np.arange(num_facets_local, dtype=np.int32)

interface_marker = 4
boundary_marker = 5

marker = np.full_like(facets, -1, dtype=np.int32)
marker[gamma_facets] = interface_marker
marker[exterior_facets] = boundary_marker
marker_filter = np.flatnonzero(marker != -1).astype(np.int32)

ft = mesh.meshtags(omega, omega.topology.dim - 1, marker_filter, marker[marker_filter])
ft.name = "interface_marker"

### Mesh visualization
We can use PyVista to visualize the computational meshes and ensure our mathematical boundaries align with our geometric expectations.

**Exercise 1.**
Run the code below to inspect the cell and boundary markers. Verify from the visualization that the domain has been correctly partitioned into an interior and exterior domain.

In [ ]:
plotter = pyvista.Plotter(shape=(1, 2), window_size=[600, 400])

plotter.subplot(0, 0)
pv_grid = pyvista.UnstructuredGrid(*plot.vtk_mesh(omega))
pv_grid.cell_data["marker"] = ct.values
plotter.add_mesh(pv_grid, categories=True, scalar_bar_args={"title": "Cell tag"})
plotter.view_xy()

plotter.subplot(0, 1)
pv_grid2 = pyvista.UnstructuredGrid(*plot.vtk_mesh(omega, 1, ft.indices))
pv_grid2.cell_data["marker"] = ft.values
plotter.add_mesh(pv_grid2, categories=True, scalar_bar_args={"title": "Facet tag"})
plotter.view_xy()

plotter.show()

**Exercise 2:** Now plot the explicitly extracted submeshes `omega_i` and `omega_e` to confirm that the geometric decomposition matches the partition indicated by the markers.

In [ ]:
pv_grid_i = pyvista.UnstructuredGrid(*plot.vtk_mesh(omega_i))
pv_grid_e = pyvista.UnstructuredGrid(*plot.vtk_mesh(omega_e))

plotter = pyvista.Plotter(window_size=[1200, 300], shape=(1, 3))
plotter.add_mesh(pv_grid_i, color="sandybrown")
plotter.subplot(0, 1)
plotter.add_mesh(pv_grid_e, color="indianred")
plotter.subplot(0, 2)
plotter.add_mesh(pv_grid_i, color="sandybrown")
plotter.add_mesh(pv_grid_e, color="indianred")
plotter.view_xy()
plotter.link_views(views=2)
plotter.show()

## Time discretization

```{exercise} Eliminate the membrane current
:label: l15-membrane-current

In the EMI model, the primary unknowns are the intracellular and extracellular potentials, $u_i$ and $u_e$. The membrane dynamics introduce an additional quantity, the membrane current $I_m$, which couples the two subdomains. In this exercise, we eliminate $I_m$ by expressing it in terms of $u_i$ and $u_e$ via time discretization of the membrane equation (5). 

To do so:

1) Discretize the membrane equation in time, i.e. consider $N_t>0$ discrete time steps $0=t_0<t_1<\cdots<t_{N_t-1}=T$, with $\Delta t = t_n - t_{n-1}$ for $n=1,...,N_t$. 

2) Approximate the time derivative $\partial/\partial_t$ using a forward finite difference scheme.

3) Treat the membrane current $I_m$ implicitly and the ionic current $I_{\text{ion}}$ explicitly.

4) Use the discrete membrane equation to derive an expression for $I_m$ in terms of $u_i$ and $u_e$.
```

## Spatial discretization with FEM

We discretize in space using a standard finite element method. The weak formulation is as follows.

Find $u_i\in V_i=V(\Omega_i)$ and $u_e\in V_e=V(\Omega_e)$ such that

$$
\begin{align}
\int_{\Omega_e} \sigma_e \nabla u_e \cdot \nabla v_e~\mathrm{d}x +
\int_\Gamma \frac{C_m}{\Delta t} (u_e - u_i) v_e ~\mathrm{d}s &=
\int_{\Omega_e} f_e v_e ~\mathrm{d}x
- \frac{C_m}{\Delta t} \int_\Gamma f v_e ~\mathrm{d}s \\
\int_{\Omega_i} \sigma_i \nabla u_i \cdot \nabla v_i~\mathrm{d}x
+ \int_\Gamma \frac{C_m}{\Delta t} (u_i - u_e) v_i ~\mathrm{d}s &=
\int_{\Omega_i} f_i v_i ~\mathrm{d}x
+ \frac{C_m}{\Delta t} \int_\Gamma f v_i ~\mathrm{d}s \\
\end{align}
$$

for all $v_e\in V_e$ and $v_i\in V_i$.

Here, $f$ groups the known, explicit terms from the previous time step $t^{n-1}$. It arises from discretizing the membrane equation (5) in time using a first-order finite difference scheme, treating the unknown membrane current implicitly ($I_m^n$) and the ionic current explicitly ($I_{\text{ion}}^{n-1}$):

$$
f = v^{n-1} - \frac{\Delta t}{C_m} I_{\text{ion}}^{n-1}.
$$

Additionally, for this specific simulation, the extracellular source is zero ($f_e=0$, and the intracellular source is our injected stimulus current ($f_i = I_{stim}$).

## Membrane model and stimulus

For this specific simulation, we apply a simple passive leak model for the membrane dynamics:

\begin{equation*}
    I_{ion} = g_m (v - E_{\text{rest}}).
\end{equation*}

Here, $g_m$ is the passive membrane conductance and $E_{\text{rest}}$ is the resting potential. 

To excite the cell and initiate a response, we will also define a temporary intracellular stimulus current $f_{stim}$ applied to the center of the $\Omega_i$ domain.

## FEniCSx implementation

To translate this math into code, we must first create integration {py:class)`Measure<ufl.Measure>`s for our domains. We use `dx` for volume integration, but we restrict it to $\Omega_i$ and $\Omega_e$ by calling the measure with the appropriate marker. *(Note: This would result in a zero integration measure if we had not passed `ct` into the initializer of `dx`!)*

In [ ]:
dx = Measure("dx", domain=omega, subdomain_data=ct)
dxI = dx(interior_marker)
dxE = dx(exterior_marker)

### Setting up the mixed function space and variational form
We create piecewise linear (CG) function spaces for both submeshes. To solve for $u_i$ and $u_e$ simultaneously, we combine these isolated spaces into a single {py:class}`MixedFunctionSpace<ufl.MixedFunctionSpace>`. From this mixed space, we can extract our coupled {py:func}`Test<ufl.TestFunctions>` and {py:func}`Trial<ufl.TrialFunctions>` functions.

In [ ]:
element = ("CG", 1)
Vi = fem.functionspace(omega_i, element)
Ve = fem.functionspace(omega_e, element)
W = MixedFunctionSpace(Vi, Ve)
vi, ve = TestFunctions(W)
ui, ue = TrialFunctions(W)


### Consistent restrictions
Next, for the interface integrals, we want to create a {py:class}`Measure<ufl.Measure>` that integrates over $\Gamma$.
However, as $\Gamma$ connects to two cells, one from $\Omega_i$ and one from $\Omega_e$, we need to define
the appropriate restrictions of $u_e$, $u_i$, $v_e$, and $v_i$ to the interface.
This is done by calling [`scifem.compute_interface_data()`](https://scientificcomputing.github.io/scifem/docs/api.html#scifem.compute_interface_data), which returns the integration data order such that the "+" side of the [restriction](https://docs.fenicsproject.org/ufl/main/manual/form_language.html#restriction-v-and-v) corresponds to the smallest cell tag of `interior_marker` and `exterior_marker`.
We pass in the ordered integration data to the initializer of a {py:class}`Measure<ufl.Measure>`.

In [ ]:
i_res = "+" if interior_marker < exterior_marker else "-"
e_res = "-" if interior_marker < exterior_marker else "+"
ordered_integration_data = scifem.compute_interface_data(ct, ft.find(interface_marker))
interface_tag = 2
dGamma = Measure(
    "dS",
    domain=omega,
    subdomain_data=[(interface_tag, ordered_integration_data.flatten())],
    subdomain_id=interface_tag,
)

We define the trace operators on the interface for our trial and test functions by applying UFL facet restrictions (+ and -).

In [ ]:
tr_ui = ui(i_res)
tr_ue = ue(e_res)
tr_vi = vi(i_res)
tr_ve = ve(e_res)

### Variational formulation
We define the problem parameters as {py:class}`Constant<dolfinx.fem.Constant>` objects.
This is to make the code efficient, even if we change the parameter values later.

In [ ]:
# Conductivities
sigma_e = fem.Constant(omega, 20.0)  # mS/cm
sigma_i = fem.Constant(omega, 5.0)  # mS/cm

# Membrane parameters
Cm = fem.Constant(omega, 1.0)  # Membrane capacitance (uF/cm^2)
g_m = fem.Constant(omega, 0.1)  # Passive membrane conductance (mS/cm^2)
E_rest = fem.Constant(omega, -85.0)  # Resting transmembrane potential (mV)

# Time stepping parameters
dt = fem.Constant(omega, 1.0e-2)  # Time step size (ms)
T_end = 1.5  # Total simulation time (ms)
num_steps = int(T_end / dt.value)

Next, we initialize the functions that will hold our solutions from the previous time step ($n$), setting the cell to its resting state.

In [ ]:
ui_n = fem.Function(Vi, name="ui_n")
ue_n = fem.Function(Ve, name="ue_n")

# Initialize the system at resting state: v = ui - ue = E_rest
ui_n.x.array[:] = E_rest.value
ue_n.x.array[:] = 0.0

# Trace evaluations of the previous step
tr_ui_n = ui_n(i_res)
tr_ue_n = ue_n(e_res)
v_n = tr_ui_n - tr_ue_n

We create a stimulus in the middle of the cell.

In [ ]:
# Define an intracellular stimulus region (a 0.1 x 0.1 square in the center)
x = SpatialCoordinate(omega)
stim_domain = conditional(
    And(And(x[0] > 0.45, x[0] < 0.55), And(x[1] > 0.45, x[1] < 0.55)), 1.0, 0.0
)
I_stim_amp = fem.Constant(omega, 0.0)  # Amplitude controlled in the time loop
I_stim = I_stim_amp * stim_domain

Finally, we can write our weak formulation in UFL.

In [ ]:
# Membrane potential: v = u_i - u_e
v_n = tr_ui_n - tr_ue_n
I_ion = g_m * (v_n - E_rest)

# Bilinear form
a = sigma_e * inner(grad(ue), grad(ve)) * dxE
a += sigma_i * inner(grad(ui), grad(vi)) * dxI
a += Cm / dt * (tr_ui - tr_ue) * (tr_vi - tr_ve) * dGamma

# Linear form
L = (Cm / dt * v_n - I_ion) * (tr_vi - tr_ve) * dGamma
L += I_stim * vi * dxI

To close the system, we apply a Dirichlet boundary condition, grounding the outer edge of the extracellular space to 0.0 mV.

In [ ]:
omega_e.topology.create_connectivity(omega_e.topology.dim - 1, omega_e.topology.dim)


# Locate the outer boundary using coordinates
def outer_boundary(x):
    return (
        np.isclose(x[0], 0.0)
        | np.isclose(x[0], 1.0)
        | np.isclose(x[1], 0.0)
        | np.isclose(x[1], 1.0)
    )


outer_facets = mesh.locate_entities_boundary(
    omega_e, omega_e.topology.dim - 1, outer_boundary
)

# Ground the outer boundary
exterior_dofs = fem.locate_dofs_topological(Ve, omega_e.topology.dim - 1, outer_facets)
bc_ground = fem.dirichletbc(default_scalar_type(0.0), exterior_dofs, Ve)
bcs = [bc_ground]

We configure the FEniCSx {py:func}`LinearProblem<dolfinx.fem.LinearProblem>`.

In [ ]:
ui_sol = fem.Function(Vi, name="ui")
ui_sol.x.array[:] = E_rest.value
ue_sol = fem.Function(Ve, name="ue")
entity_maps = [interior_to_parent, exterior_to_parent]

petsc_options = {
    "ksp_type": "cg",
    "pc_type": "hypre",
    "ksp_rtol": 1e-6,
}

problem = fem.petsc.LinearProblem(
    extract_blocks(a),
    extract_blocks(L),
    u=[ui_sol, ue_sol],
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="emi_passive_",
    entity_maps=entity_maps,
)

### Simulation and visualization

Before running the time-stepping loop, we set up a PyVista plotter to capture an animated GIF of our simulation.

In [ ]:
import pyvista
from IPython.display import Image

plotter = pyvista.Plotter(window_size=[800, 600], off_screen=True)
plotter.open_gif("output/EMI_single_cell.gif", fps=5)

# Setup intracellular grid
topology_i, cell_types_i, geometry_i = plot.vtk_mesh(Vi)
grid_i = pyvista.UnstructuredGrid(topology_i, cell_types_i, geometry_i)
grid_i.point_data["u_i"] = ui_n.x.array

# Setup extracellular grid
topology_e, cell_types_e, geometry_e = plot.vtk_mesh(Ve)
grid_e = pyvista.UnstructuredGrid(topology_e, cell_types_e, geometry_e)
grid_e.point_data["u_e"] = ue_n.x.array

# Add intracellular mesh
plotter.add_mesh(
    grid_i,
    scalars="u_i",
    cmap="viridis",
    clim=[-85.0, -80.0],
    scalar_bar_args={
        "title": "u_i [mV]",
        "position_x": 0.05,
        "position_y": 0.05,
        "width": 0.4,
    },
)

# Add extracellular mesh
plotter.add_mesh(
    grid_e,
    scalars="u_e",
    cmap="plasma",
    clim=[-0.01, 0.1],
    scalar_bar_args={
        "title": "u_e [mV]",
        "position_x": 0.55,
        "position_y": 0.05,
        "width": 0.4,
    },
)

plotter.view_xy()

# Add initial time label
t = 0.0
plotter.add_text(
    f"Time: {t:.3f} ms",
    name="time_label",
    font_size=14,
)
plotter.write_frame()

Now, we run the time-stepping loop! We will trigger a stimulus between $t=0.5$ and $t=1.5$ ms to spark the passive membrane, saving the data to VTX files and writing the frames to our GIF.

In [ ]:
# Setup VTX writers and write initial state
output_dir = Path("output")
output_dir.mkdir(exist_ok=True, parents=True)
writer_i = io.VTXWriter(omega_i.comm, output_dir / "EMI_singlecell_ui.bp", [ui_sol])
writer_e = io.VTXWriter(omega_e.comm, output_dir / "EMI_singlecell_ue.bp", [ue_sol])
writer_i.write(0.0)
writer_e.write(0.0)

# Run simulation
t = 0.0
for i in range(num_steps):
    t += dt.value

    # Toggle stimulus
    if 0.1 <= t <= 1.0:
        I_stim_amp.value = 1000.0
    else:
        I_stim_amp.value = 0.0

    # Solve the linear system
    problem.solve()

    # Update previous step solutions
    ui_n.interpolate(ui_sol)
    ue_n.interpolate(ue_sol)

    if i % 10 == 0:
        # Write current time step to files
        writer_i.write(t)
        writer_e.write(t)

        # Update PyVista arrays and time label for current frame
        grid_i["u_i"][:] = ui_sol.x.array
        grid_e["u_e"][:] = ue_sol.x.array
        plotter.add_text(
            f"Time: {t:.3f} ms",
            name="time_label",
            font_size=14,
        )
        plotter.write_frame()

plotter.close()
writer_i.close()
writer_e.close()

print("Simulation complete.")

Run the cell below to view the final animation.

In [ ]:
# Display GIF
Image(filename="output/EMI_single_cell.gif")

## References
```{bibliography}
   :filter: cited
   :labelprefix:
   :keyprefix: emisinglecell-
```